# Phase 7 — Explainability and Advanced Model Evaluation

This notebook analyses the behaviour of the final obesity-risk classification pipeline selected during Phase 6.

The final model is treated as frozen. No hyperparameter tuning or model selection is performed using the test data during this phase.

The analysis focuses on model explainability, prediction behaviour, ordinal errors, controlled feature-ablation experiments, and subgroup performance.

In [1]:
from pathlib import Path
import json
import sys

import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

In [2]:
current_path = Path.cwd().resolve()

PROJECT_ROOT = current_path

while not (
    PROJECT_ROOT
    / "src"
    / "preprocessing.py"
).exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError(
            "Could not locate the project root"
        )

    PROJECT_ROOT = PROJECT_ROOT.parent

print(
    "Project root:",
    PROJECT_ROOT,
)

Project root: C:\Users\User\OneDrive\Desktop\Obesity-Risk-Intelligence-System


In [3]:
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )

In [4]:
from src.preprocessing import (
    PREDICTIVE_FEATURES,
)

In [5]:
print(
    "Predictive feature count:",
    len(PREDICTIVE_FEATURES),
)

PREDICTIVE_FEATURES

Predictive feature count: 16


['Age',
 'Height',
 'Weight',
 'FCVC',
 'NCP',
 'CH2O',
 'FAF',
 'TUE',
 'CAEC',
 'CALC',
 'Gender',
 'family_history_with_overweight',
 'FAVC',
 'SMOKE',
 'SCC',
 'MTRANS']

In [6]:
DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "obesity.csv"
)

FINAL_MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "obesity_risk_pipeline.joblib"
)

MODEL_METADATA_PATH = (
    PROJECT_ROOT
    / "models"
    / "model_metadata.json"
)

REPORT_DIRECTORY = (
    PROJECT_ROOT
    / "reports"
    / "generated"
)

In [7]:
print(
    "Dataset exists:",
    DATASET_PATH.exists(),
)

print(
    "Final model exists:",
    FINAL_MODEL_PATH.exists(),
)

print(
    "Metadata exists:",
    MODEL_METADATA_PATH.exists(),
)

Dataset exists: True
Final model exists: True
Metadata exists: True


In [8]:
final_model_size_bytes = (
    FINAL_MODEL_PATH.stat().st_size
)

final_model_size_mb = (
    final_model_size_bytes
    / (1024 * 1024)
)

print(
    "Final model size:",
    final_model_size_bytes,
    "bytes",
)

print(
    "Final model size:",
    round(
        final_model_size_mb,
        3,
    ),
    "MB",
)

Final model size: 919933 bytes
Final model size: 0.877 MB


In [9]:
with open(
    MODEL_METADATA_PATH,
    "r",
    encoding="utf-8",
) as metadata_file:
    model_metadata = json.load(
        metadata_file
    )

model_metadata

{'project': 'Obesity Risk Intelligence System',
 'selected_candidate': 'Tuned Gradient Boosting',
 'model_family': 'Gradient Boosting',
 'configuration': 'Tuned',
 'random_state': 42,
 'predictive_feature_count': 16,
 'transformed_feature_count': 25,
 'target_class_count': 7,
 'predictive_features': ['Age',
  'Height',
  'Weight',
  'FCVC',
  'NCP',
  'CH2O',
  'FAF',
  'TUE',
  'CAEC',
  'CALC',
  'Gender',
  'family_history_with_overweight',
  'FAVC',
  'SMOKE',
  'SCC',
  'MTRANS'],
 'target_classes': ['Insufficient_Weight',
  'Normal_Weight',
  'Overweight_Level_I',
  'Overweight_Level_II',
  'Obesity_Type_I',
  'Obesity_Type_II',
  'Obesity_Type_III'],
 'development_records': 17644,
 'test_records': 3114,
 'selection_validation_macro_f1': 0.8977760050085329,
 'final_test_accuracy': 0.9075144508670521,
 'final_test_balanced_accuracy': 0.8976420939847197,
 'final_test_macro_f1': 0.8973281823370796,
 'final_test_weighted_f1': 0.9072142069759439,
 'scikit_learn_version': '1.8.0'}

In [10]:
metadata_summary = {
    "Selected Candidate": (
        model_metadata[
            "selected_candidate"
        ]
    ),
    "Model Family": (
        model_metadata[
            "model_family"
        ]
    ),
    "Configuration": (
        model_metadata[
            "configuration"
        ]
    ),
    "Predictive Features": (
        model_metadata[
            "predictive_feature_count"
        ]
    ),
    "Target Classes": (
        model_metadata[
            "target_class_count"
        ]
    ),
    "Development Records": (
        model_metadata[
            "development_records"
        ]
    ),
    "Test Records": (
        model_metadata[
            "test_records"
        ]
    ),
}

pd.Series(
    metadata_summary,
    name="Value",
)

Selected Candidate     Tuned Gradient Boosting
Model Family                 Gradient Boosting
Configuration                            Tuned
Predictive Features                         16
Target Classes                               7
Development Records                      17644
Test Records                              3114
Name: Value, dtype: object

In [11]:
metadata_predictive_features = (
    model_metadata[
        "predictive_features"
    ]
)

print(
    "Metadata feature count:",
    len(metadata_predictive_features),
)

print(
    "Code feature count:",
    len(PREDICTIVE_FEATURES),
)

Metadata feature count: 16
Code feature count: 16


In [12]:
assert (
    list(PREDICTIVE_FEATURES)
    == metadata_predictive_features
)

print(
    "Metadata and preprocessing feature "
    "configuration match"
)

Metadata and preprocessing feature configuration match


In [13]:
final_model = joblib.load(
    FINAL_MODEL_PATH
)

final_model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numerical', ...), ('ordinal', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the diffe

In [14]:
print(
    "Loaded model type:",
    type(final_model).__name__,
)

Loaded model type: Pipeline


In [15]:
assert isinstance(
    final_model,
    Pipeline,
)

print(
    "Final model is a valid sklearn Pipeline"
)

Final model is a valid sklearn Pipeline


In [16]:
print(
    "Pipeline steps:"
)

for step_name, step_object in (
    final_model.steps
):
    print(
        f"- {step_name}: "
        f"{type(step_object).__name__}"
    )

Pipeline steps:
- preprocessor: ColumnTransformer
- classifier: GradientBoostingClassifier


In [17]:
final_preprocessor = (
    final_model
    .named_steps[
        "preprocessor"
    ]
)

final_classifier = (
    final_model
    .named_steps[
        "classifier"
    ]
)

print(
    "Preprocessor:",
    type(
        final_preprocessor
    ).__name__,
)

print(
    "Classifier:",
    type(
        final_classifier
    ).__name__,
)

Preprocessor: ColumnTransformer
Classifier: GradientBoostingClassifier


In [18]:
transformed_feature_names = (
    final_preprocessor
    .get_feature_names_out()
)

print(
    "Raw feature count:",
    len(PREDICTIVE_FEATURES),
)

print(
    "Transformed feature count:",
    len(
        transformed_feature_names
    ),
)

Raw feature count: 16
Transformed feature count: 25


In [19]:
for feature_name in (
    transformed_feature_names
):
    print(
        feature_name
    )

numerical__Age
numerical__Height
numerical__Weight
numerical__FCVC
numerical__NCP
numerical__CH2O
numerical__FAF
numerical__TUE
ordinal__CAEC
ordinal__CALC
nominal__Gender_Female
nominal__Gender_Male
nominal__family_history_with_overweight_no
nominal__family_history_with_overweight_yes
nominal__FAVC_no
nominal__FAVC_yes
nominal__SMOKE_no
nominal__SMOKE_yes
nominal__SCC_no
nominal__SCC_yes
nominal__MTRANS_Automobile
nominal__MTRANS_Bike
nominal__MTRANS_Motorbike
nominal__MTRANS_Public_Transportation
nominal__MTRANS_Walking


In [20]:
assert (
    len(transformed_feature_names)
    == model_metadata[
        "transformed_feature_count"
    ]
)

print(
    "Transformed feature count matches metadata"
)

Transformed feature count matches metadata


In [21]:
model_classes = list(
    final_classifier.classes_
)

print(
    "Model classes:"
)

for class_name in model_classes:
    print(
        "-",
        class_name,
    )

Model classes:
- Insufficient_Weight
- Normal_Weight
- Obesity_Type_I
- Obesity_Type_II
- Obesity_Type_III
- Overweight_Level_I
- Overweight_Level_II


In [22]:
metadata_classes = (
    model_metadata[
        "target_classes"
    ]
)

print(
    "Model class count:",
    len(model_classes),
)

print(
    "Metadata class count:",
    len(metadata_classes),
)

Model class count: 7
Metadata class count: 7


In [23]:
assert set(
    model_classes
) == set(
    metadata_classes
)

print(
    "Model classes match metadata"
)

Model classes match metadata


In [24]:
df = pd.read_csv(
    DATASET_PATH
)

print(
    "Dataset shape:",
    df.shape,
)

df.head()

Dataset shape: (20758, 18)


,id,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,NObeyesdad
0,0,Male,24.443011,1.699998,81.669950,yes,yes,2.000000,2.983297,Sometimes,no,2.763573,no,0.000000,0.976473,Sometimes,Public_Transportation,Overweight_Level_II
1,1,Female,18.000000,1.560000,57.000000,yes,yes,2.000000,3.000000,Frequently,no,2.000000,no,1.000000,1.000000,no,Automobile,Normal_Weight
2,2,Female,18.000000,1.711460,50.165754,yes,yes,1.880534,1.411685,Sometimes,no,1.910378,no,0.866045,1.673584,no,Public_Transportation,Insufficient_Weight
3,3,Female,20.952737,1.710730,131.274851,yes,yes,3.000000,3.000000,Sometimes,no,1.674061,no,1.467863,0.780199,Sometimes,Public_Transportation,Obesity_Type_III
4,4,Male,31.641081,1.914186,93.798055,yes,yes,2.679664,1.971472,Sometimes,no,1.979848,no,1.967973,0.931721,Sometimes,Public_Transportation,Overweight_Level_II


In [25]:
TARGET_COLUMN = "NObeyesdad"
RANDOM_STATE = 42

In [26]:
X = df[
    PREDICTIVE_FEATURES
].copy()

y = df[
    TARGET_COLUMN
].copy()

In [27]:
print(
    "Feature shape:",
    X.shape,
)

print(
    "Target shape:",
    y.shape,
)

Feature shape: (20758, 16)
Target shape: (20758,)


In [28]:
X_train, X_temp, y_train, y_temp = (
    train_test_split(
        X,
        y,
        test_size=0.30,
        random_state=RANDOM_STATE,
        stratify=y,
    )
)

In [29]:
(
    X_validation,
    X_test,
    y_validation,
    y_test,
) = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_temp,
)

In [30]:
print(
    "Training records:",
    len(X_train),
)

print(
    "Validation records:",
    len(X_validation),
)

print(
    "Test records:",
    len(X_test),
)

Training records: 14530
Validation records: 3114
Test records: 3114


In [31]:
assert len(
    X_train
) == 14530

assert len(
    X_validation
) == 3114

assert len(
    X_test
) == 3114

assert len(
    y_test
) == 3114

print(
    "Original dataset split recreated successfully"
)

Original dataset split recreated successfully


In [32]:
train_validation_overlap = (
    X_train.index.intersection(
        X_validation.index
    )
)

train_test_overlap = (
    X_train.index.intersection(
        X_test.index
    )
)

validation_test_overlap = (
    X_validation.index.intersection(
        X_test.index
    )
)

print(
    "Train-validation overlap:",
    len(
        train_validation_overlap
    ),
)

print(
    "Train-test overlap:",
    len(
        train_test_overlap
    ),
)

print(
    "Validation-test overlap:",
    len(
        validation_test_overlap
    ),
)

Train-validation overlap: 0
Train-test overlap: 0
Validation-test overlap: 0


In [33]:
assert len(
    X_test
) == model_metadata[
    "test_records"
]

print(
    "Recreated test dataset matches metadata size"
)

Recreated test dataset matches metadata size


In [34]:
phase7_test_predictions = (
    final_model.predict(
        X_test
    )
)

In [35]:
print(
    "Predictions generated:",
    len(
        phase7_test_predictions
    ),
)

Predictions generated: 3114


In [36]:
phase7_test_probabilities = (
    final_model.predict_proba(
        X_test
    )
)

print(
    "Probability matrix shape:",
    phase7_test_probabilities.shape,
)

Probability matrix shape: (3114, 7)


In [37]:
assert np.allclose(
    phase7_test_probabilities.sum(
        axis=1
    ),
    1.0,
)

print(
    "Prediction probabilities verified"
)

Prediction probabilities verified


In [38]:
example_probability_row = pd.Series(
    phase7_test_probabilities[0],
    index=model_classes,
    name="Probability",
)

example_probability_row.sort_values(
    ascending=False
)

Obesity_Type_III       0.999160
Obesity_Type_I         0.000735
Overweight_Level_II    0.000042
Overweight_Level_I     0.000029
Obesity_Type_II        0.000028
Normal_Weight          0.000005
Insufficient_Weight    0.000001
Name: Probability, dtype: float64

In [39]:
example_predicted_class = (
    phase7_test_predictions[0]
)

example_highest_probability_class = (
    model_classes[
        np.argmax(
            phase7_test_probabilities[
                0
            ]
        )
    ]
)

print(
    "Predicted class:",
    example_predicted_class,
)

print(
    "Highest-probability class:",
    example_highest_probability_class,
)

Predicted class: Obesity_Type_III
Highest-probability class: Obesity_Type_III


In [40]:
assert (
    example_predicted_class
    == example_highest_probability_class
)

print(
    "Prediction/probability consistency verified"
)

Prediction/probability consistency verified


In [41]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)

In [42]:
phase7_verification_metrics = {
    "Accuracy": (
        accuracy_score(
            y_test,
            phase7_test_predictions,
        )
    ),
    "Balanced Accuracy": (
        balanced_accuracy_score(
            y_test,
            phase7_test_predictions,
        )
    ),
    "Macro F1": (
        f1_score(
            y_test,
            phase7_test_predictions,
            average="macro",
            zero_division=0,
        )
    ),
    "Weighted F1": (
        f1_score(
            y_test,
            phase7_test_predictions,
            average="weighted",
            zero_division=0,
        )
    ),
}

pd.Series(
    phase7_verification_metrics,
    name="Score",
).round(4)

Accuracy             0.9075
Balanced Accuracy    0.8976
Macro F1             0.8973
Weighted F1          0.9072
Name: Score, dtype: float64

In [43]:
saved_phase6_metrics = {
    "Accuracy": (
        model_metadata[
            "final_test_accuracy"
        ]
    ),
    "Balanced Accuracy": (
        model_metadata[
            "final_test_balanced_accuracy"
        ]
    ),
    "Macro F1": (
        model_metadata[
            "final_test_macro_f1"
        ]
    ),
    "Weighted F1": (
        model_metadata[
            "final_test_weighted_f1"
        ]
    ),
}

verification_comparison_df = pd.DataFrame(
    {
        "Phase 6 Saved": (
            saved_phase6_metrics
        ),
        "Phase 7 Reloaded": (
            phase7_verification_metrics
        ),
    }
)

verification_comparison_df[
    "Difference"
] = (
    verification_comparison_df[
        "Phase 7 Reloaded"
    ]
    - verification_comparison_df[
        "Phase 6 Saved"
    ]
)

verification_comparison_df.round(8)

,Phase 6 Saved,Phase 7 Reloaded,Difference
Accuracy,0.907514,0.907514,0.0
Balanced Accuracy,0.897642,0.897642,0.0
Macro F1,0.897328,0.897328,0.0
Weighted F1,0.907214,0.907214,0.0


In [44]:
for metric_name in (
    saved_phase6_metrics
):
    assert np.isclose(
        saved_phase6_metrics[
            metric_name
        ],
        phase7_verification_metrics[
            metric_name
        ],
    )

In [45]:
test_analysis_df = (
    X_test.copy()
)

test_analysis_df[
    "True_Class"
] = y_test

test_analysis_df[
    "Predicted_Class"
] = (
    phase7_test_predictions
)

test_analysis_df[
    "Correct_Prediction"
] = (
    test_analysis_df[
        "True_Class"
    ]
    == test_analysis_df[
        "Predicted_Class"
    ]
)

In [46]:
test_analysis_df.head()

,Age,Height,Weight,FCVC,NCP,CH2O,FAF,TUE,CAEC,CALC,Gender,family_history_with_overweight,FAVC,SMOKE,SCC,MTRANS,True_Class,Predicted_Class,Correct_Prediction
16056,26.000000,1.617390,105.448264,3.000000,3.0,1.031354,0.000000,0.374650,Sometimes,Sometimes,Female,yes,yes,no,no,Public_Transportation,Obesity_Type_III,Obesity_Type_III,True
19997,27.635029,1.848553,120.998266,2.543563,3.0,2.436990,1.042680,0.474836,Sometimes,Sometimes,Male,yes,yes,no,no,Public_Transportation,Obesity_Type_II,Obesity_Type_II,True
16512,18.000000,1.560000,50.000000,2.000000,4.0,2.000000,2.000000,1.000000,Frequently,Sometimes,Female,no,yes,no,no,Automobile,Insufficient_Weight,Normal_Weight,False
344,37.936044,1.750150,118.668332,2.457548,3.0,2.104264,0.598655,0.000000,Sometimes,Sometimes,Male,yes,yes,no,no,Automobile,Obesity_Type_I,Obesity_Type_II,False
6633,19.000000,1.510000,45.000000,3.000000,3.0,1.000000,1.000000,1.000000,Frequently,no,Female,no,no,no,no,Public_Transportation,Insufficient_Weight,Insufficient_Weight,True


In [47]:
test_analysis_df[
    "Prediction_Confidence"
] = (
    phase7_test_probabilities.max(
        axis=1
    )
)

test_analysis_df[
    [
        "True_Class",
        "Predicted_Class",
        "Correct_Prediction",
        "Prediction_Confidence",
    ]
].head()

,True_Class,Predicted_Class,Correct_Prediction,Prediction_Confidence
16056,Obesity_Type_III,Obesity_Type_III,True,0.999160
19997,Obesity_Type_II,Obesity_Type_II,True,0.996750
16512,Insufficient_Weight,Normal_Weight,False,0.763893
344,Obesity_Type_I,Obesity_Type_II,False,0.990206
6633,Insufficient_Weight,Insufficient_Weight,True,0.879578


In [48]:
prediction_correctness_counts = (
    test_analysis_df[
        "Correct_Prediction"
    ]
    .value_counts()
)

prediction_correctness_counts

Correct_Prediction
True     2826
False     288
Name: count, dtype: int64

In [49]:
test_analysis_df

,Age,Height,Weight,FCVC,NCP,CH2O,FAF,TUE,CAEC,CALC,Gender,family_history_with_overweight,FAVC,SMOKE,SCC,MTRANS,True_Class,Predicted_Class,Correct_Prediction,Prediction_Confidence
16056,26.000000,1.617390,105.448264,3.000000,3.0,1.031354,0.000000,0.374650,Sometimes,Sometimes,Female,yes,yes,no,no,Public_Transportation,Obesity_Type_III,Obesity_Type_III,True,0.999160
19997,27.635029,1.848553,120.998266,2.543563,3.0,2.436990,1.042680,0.474836,Sometimes,Sometimes,Male,yes,yes,no,no,Public_Transportation,Obesity_Type_II,Obesity_Type_II,True,0.996750
16512,18.000000,1.560000,50.000000,2.000000,4.0,2.000000,2.000000,1.000000,Frequently,Sometimes,Female,no,yes,no,no,Automobile,Insufficient_Weight,Normal_Weight,False,0.763893
344,37.936044,1.750150,118.668332,2.457548,3.0,2.104264,0.598655,0.000000,Sometimes,Sometimes,Male,yes,yes,no,no,Automobile,Obesity_Type_I,Obesity_Type_II,False,0.990206
6633,19.000000,1.510000,45.000000,3.000000,3.0,1.000000,1.000000,1.000000,Frequently,no,Female,no,no,no,no,Public_Transportation,Insufficient_Weight,Insufficient_Weight,True,0.879578
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17983,26.957645,1.817231,120.202596,3.000000,3.0,2.966647,1.624981,0.447224,Sometimes,Sometimes,Male,yes,yes,no,no,Public_Transportation,Obesity_Type_II,Obesity_Type_II,True,0.983314
15761,18.000000,1.739450,51.457226,1.889199,3.0,1.959531,0.866045,1.887386,Sometimes,Sometimes,Male,yes,yes,no,no,Public_Transportation,Insufficient_Weight,Insufficient_Weight,True,0.992413
3177,39.126310,1.562889,77.473204,2.000000,1.0,1.959531,0.000000,0.000000,Sometimes,Sometimes,Female,yes,yes,no,no,Automobile,Obesity_Type_I,Obesity_Type_I,True,0.974554
189,25.765628,1.629225,104.768318,3.000000,3.0,2.682909,0.000000,0.629285,Sometimes,Sometimes,Female,yes,yes,no,no,Public_Transportation,Obesity_Type_III,Obesity_Type_III,True,0.998610


In [50]:
assert FINAL_MODEL_PATH.exists()

assert MODEL_METADATA_PATH.exists()

assert isinstance(
    final_model,
    Pipeline,
)

assert (
    list(PREDICTIVE_FEATURES)
    == metadata_predictive_features
)

assert len(
    PREDICTIVE_FEATURES
) == 16

assert len(
    transformed_feature_names
) == 25

assert len(
    model_classes
) == 7

assert set(
    model_classes
) == set(
    metadata_classes
)

assert df.shape == (
    20758,
    18,
)

assert len(
    X_test
) == 3114

assert len(
    phase7_test_predictions
) == 3114

assert (
    phase7_test_probabilities.shape
    == (
        3114,
        7,
    )
)

assert np.allclose(
    phase7_test_probabilities.sum(
        axis=1
    ),
    1.0,
)

assert len(
    train_test_overlap
) == 0

assert len(
    validation_test_overlap
) == 0

assert len(
    test_analysis_df
) == 3114

print(
    "Validation passed"
)

Validation passed
